In [1]:

# ============================================================================
# CELL 1: Install Dependencies + Restart Kernel
# ============================================================================
 
import os
 
print("Checking GPU...")
os.system('nvidia-smi')
 
print("\nInstalling dependencies...")
print("="*80)
 
os.system("pip install -q transformers datasets accelerate scikit-learn")
 
print("✓ Dependencies installed")
print("\n*** KERNEL IS RESTARTING — wait for it, then run cells 2 onwards ***")
 
import IPython
IPython.Application.instance().kernel.do_shutdown(True)
 
 

Checking GPU...
Thu Mar 19 10:02:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------

{'status': 'ok', 'restart': True}

In [1]:
# ============================================================================
# CELL 2: Import Libraries
# ============================================================================
 
import os
import json
import numpy as np
import pandas as pd
import torch
import warnings
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
 
warnings.filterwarnings('ignore')
 
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
 

PyTorch version: 2.9.0+cu126
CUDA available: True
CUDA device: Tesla T4


In [2]:
# ============================================================================
# CELL 3: Configuration
# ============================================================================
 
class Config:
    """Training configuration"""
 
    # Dataset path — UPDATE THIS with your Kaggle dataset name
    DATASET_PATH = "/kaggle/input/datasets/nitishpatil2027/stance-detection-fever-dataset"  # ← CHANGE THIS
 
    # Model
    MODEL_NAME = "bert-base-cased"  # cased is important for stance detection
    MAX_LENGTH = 256                 # longer for claim + evidence
 
    # Training
    BATCH_SIZE = 16
    LEARNING_RATE = 2e-5
    NUM_EPOCHS = 3
    WARMUP_STEPS = 500
    WEIGHT_DECAY = 0.01
 
    # Output
    OUTPUT_DIR = "./stance_detector_output"
    SEED = 42
 
config = Config()
 
print("\nConfiguration:")
print(f"  Model:         {config.MODEL_NAME}")
print(f"  Max length:    {config.MAX_LENGTH}")
print(f"  Batch size:    {config.BATCH_SIZE}")
print(f"  Epochs:        {config.NUM_EPOCHS}")
print(f"  Learning rate: {config.LEARNING_RATE}")
 
 


Configuration:
  Model:         bert-base-cased
  Max length:    256
  Batch size:    16
  Epochs:        3
  Learning rate: 2e-05


In [3]:
# ============================================================================
# CELL 4: Load Dataset
# ============================================================================
 
print("\nLoading dataset...")
print("="*80)
 
train_df = pd.read_csv(f"{config.DATASET_PATH}/stance_detection_train.csv")
val_df   = pd.read_csv(f"{config.DATASET_PATH}/stance_detection_val.csv")
test_df  = pd.read_csv(f"{config.DATASET_PATH}/stance_detection_test.csv")
 
print(f"Train: {len(train_df):,} examples")
print(f"Val:   {len(val_df):,} examples")
print(f"Test:  {len(test_df):,} examples")
 
# Load label mapping
with open(f"{config.DATASET_PATH}/stance_labels.json", 'r') as f:
    label_info = json.load(f)
 
label_names = label_info['labels']
label2id    = label_info['label2id']
id2label    = {int(k): v for k, v in label_info['id2label'].items()}
num_labels  = label_info['num_labels']
 
print(f"\nLabels ({num_labels}):")
for label in label_names:
    print(f"  {label2id[label]}: {label}")
 
print("\nLabel distribution (Train):")
for label in label_names:
    count = (train_df['label'] == label).sum()
    print(f"  {label}: {count:,} ({count/len(train_df)*100:.1f}%)")
 
print("\nSample claims:")
for label in label_names:
    sample = train_df[train_df['label'] == label].iloc[0]
    print(f"\n  {label}:")
    print(f"    Claim:    {sample['claim'][:80]}...")
    print(f"    Evidence: {sample['evidence'][:80]}...")
 
print("\n✓ Dataset loaded successfully")
 


Loading dataset...
Train: 208,346 examples
Val:   9,999 examples
Test:  9,999 examples

Labels (3):
  0: SUPPORTS
  1: REFUTES
  2: NOT ENOUGH INFO

Label distribution (Train):
  SUPPORTS: 123,594 (59.3%)
  REFUTES: 49,113 (23.6%)
  NOT ENOUGH INFO: 35,639 (17.1%)

Sample claims:

  SUPPORTS:
    Claim:    The Fox Broadcasting Company ( often shortened to Fox and stylized as FOX ) is a...
    Evidence: Nikolaj Coster-Waldau worked with the Fox Broadcasting Company....

  REFUTES:
    Claim:    Adrienne Eliza Houghton ( née Bailon ; born October 24 , 1983 ) is an American ...
    Evidence: Adrienne Bailon is an accountant....

  NOT ENOUGH INFO:
    Claim:    System of a Down . The group briefly disbanded in August 2006 and reunited in No...
    Evidence: System of a Down briefly disbanded in limbo....

✓ Dataset loaded successfully


In [4]:
# ============================================================================
# CELL 5: PyTorch Dataset Class
# ============================================================================
 
class StanceDataset(Dataset):
    """
    Encodes claim + evidence as sentence pair:
    [CLS] claim [SEP] evidence [SEP]
    """
 
    def __init__(self, claims, evidence, labels, tokenizer, label2id, max_length):
        self.claims   = claims
        self.evidence = evidence
        self.labels   = labels
        self.tokenizer = tokenizer
        self.label2id  = label2id
        self.max_length = max_length
 
    def __len__(self):
        return len(self.claims)
 
    def __getitem__(self, idx):
        claim = str(self.claims[idx])
        evid  = str(self.evidence[idx])
        label = self.labels[idx]
 
        # Proper sentence-pair encoding
        encoding = self.tokenizer(
            claim,
            evid,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
 
        return {
            'input_ids':      encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels':         torch.tensor(self.label2id[label], dtype=torch.long)
        }
 
print("✓ Dataset class defined")
 

✓ Dataset class defined


In [5]:
# ============================================================================
# CELL 6: Load Tokenizer and Create Datasets
# ============================================================================
 
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
 
print("Creating datasets...")
 
train_dataset = StanceDataset(
    train_df['claim'].values,
    train_df['evidence'].values,
    train_df['label'].values,
    tokenizer, label2id, config.MAX_LENGTH
)
 
val_dataset = StanceDataset(
    val_df['claim'].values,
    val_df['evidence'].values,
    val_df['label'].values,
    tokenizer, label2id, config.MAX_LENGTH
)
 
test_dataset = StanceDataset(
    test_df['claim'].values,
    test_df['evidence'].values,
    test_df['label'].values,
    tokenizer, label2id, config.MAX_LENGTH
)
 
print(f"✓ Datasets created:")
print(f"  Train: {len(train_dataset):,}")
print(f"  Val:   {len(val_dataset):,}")
print(f"  Test:  {len(test_dataset):,}")
 
# Verify encoding
sample = train_dataset[0]
print(f"\nSample encoding:")
print(f"  input_ids shape:      {sample['input_ids'].shape}")
print(f"  attention_mask shape: {sample['attention_mask'].shape}")
print(f"  label:                {sample['labels'].item()}")
 
 


Loading tokenizer...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Creating datasets...
✓ Datasets created:
  Train: 208,346
  Val:   9,999
  Test:  9,999

Sample encoding:
  input_ids shape:      torch.Size([256])
  attention_mask shape: torch.Size([256])
  label:                0


In [6]:
# ============================================================================
# CELL 7: Load Model
# ============================================================================
 
print("\nLoading model...")
 
model = AutoModelForSequenceClassification.from_pretrained(
    config.MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
 
print(f"✓ Model loaded on {device}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
 
 


Loading model...


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model loaded on cuda
  Parameters: 108,312,579


In [7]:
# ============================================================================
# CELL 8: Metrics
# ============================================================================
 
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
 
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    acc = accuracy_score(labels, predictions)
 
    per_class_f1 = precision_recall_fscore_support(
        labels, predictions, average=None
    )[2]
 
    metrics = {
        'accuracy':  acc,
        'precision': precision,
        'recall':    recall,
        'f1':        f1,
    }
 
    for i, label in enumerate(label_names):
        key = f"f1_{label.lower().replace(' ', '_')}"
        metrics[key] = per_class_f1[i]
 
    return metrics
 
print("✓ Metrics function defined")

✓ Metrics function defined


In [8]:
# ============================================================================
# CELL 9: Training Arguments
# ============================================================================
 
# Auto-detect correct parameter name for this transformers version
import transformers
transformers_version = tuple(int(x) for x in transformers.__version__.split(".")[:2])
eval_strategy_key = "evaluation_strategy" if transformers_version < (4, 41) else "eval_strategy"
 
print(f"Transformers version: {transformers.__version__}")
print(f"Using eval param:     {eval_strategy_key}")
 
training_args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
 
    # Training hyperparameters
    num_train_epochs=config.NUM_EPOCHS,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE * 2,
    learning_rate=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY,
    warmup_steps=config.WARMUP_STEPS,
 
    # Logging
    logging_dir=f"{config.OUTPUT_DIR}/logs",
    logging_steps=100,
 
    # Evaluation — version-safe
    **{eval_strategy_key: "steps"},
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
 
    # Other
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
    seed=config.SEED,
)
 
print("✓ Training arguments configured")
 

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Transformers version: 5.2.0
Using eval param:     eval_strategy
✓ Training arguments configured


In [9]:
# ============================================================================
# CELL 10: Initialize Trainer
# ============================================================================
 
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)
 
print("✓ Trainer initialized")
 

✓ Trainer initialized


In [10]:
# ============================================================================
# CELL 11: Train (2-3 HOURS)
# ============================================================================
 
print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)
print("This will take approximately 2-3 hours on T4 GPU")
print("="*80 + "\n")
 
trainer.train()
 
print("\n" + "="*80)
print("✓ TRAINING COMPLETE!")
print("="*80)
 
 


STARTING TRAINING
This will take approximately 2-3 hours on T4 GPU



Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Supports,F1 Refutes,F1 Not Enough Info
500,0.980382,1.756511,0.672267,0.693702,0.672267,0.666803,0.749180,0.631093,0.620135
1000,0.908503,1.815639,0.704070,0.705683,0.704070,0.698384,0.772512,0.710435,0.612205
1500,0.804387,1.808635,0.710771,0.713223,0.710771,0.706136,0.776792,0.717297,0.624318
2000,0.774471,1.769168,0.706271,0.713404,0.706271,0.699434,0.768428,0.720697,0.609178
2500,0.727085,1.774246,0.716272,0.719478,0.716272,0.711201,0.784747,0.720881,0.627974
3000,0.731945,1.819900,0.713871,0.722889,0.713871,0.709199,0.776787,0.719799,0.631011
3500,0.767353,1.884968,0.719372,0.723934,0.719372,0.714835,0.785057,0.724922,0.634527
4000,0.760821,1.737286,0.725173,0.723963,0.725173,0.719646,0.793840,0.739768,0.625331
4500,0.733649,1.762823,0.730673,0.731080,0.730673,0.726288,0.794723,0.743843,0.640297
5000,0.689255,1.785599,0.727973,0.728921,0.727973,0.723162,0.791450,0.742310,0.635727


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


✓ TRAINING COMPLETE!


In [11]:
# ============================================================================
# CELL 12: Evaluate on Test Set
# ============================================================================
 
print("\nEvaluating on test set...")
print("="*80)
 
test_results = trainer.evaluate(test_dataset)
 
print("\nTEST SET RESULTS:")
print("="*80)
print(f"Accuracy:  {test_results['eval_accuracy']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"Recall:    {test_results['eval_recall']:.4f}")
print(f"F1 Score:  {test_results['eval_f1']:.4f}")
 
print("\nPer-class F1 scores:")
for label in label_names:
    key = f"eval_f1_{label.lower().replace(' ', '_')}"
    if key in test_results:
        print(f"  {label}: {test_results[key]:.4f}")
 
print("="*80)
 
# Confusion matrix
predictions  = trainer.predict(test_dataset)
pred_labels  = np.argmax(predictions.predictions, axis=1)
true_labels  = [label2id[label] for label in test_df['label'].values]
 
cm = confusion_matrix(true_labels, pred_labels)
 
print("\nConfusion Matrix:")
print(f"{'':25}", end='')
for label in label_names:
    print(f"{label[:15]:>16}", end='')
print()
for i, label in enumerate(label_names):
    print(f"{label[:25]:25}", end='')
    for j in range(len(label_names)):
        print(f"{cm[i][j]:>16}", end='')
    print()
 
print("\nDetailed Classification Report:")
print("="*80)
true_label_names = [id2label[i] for i in true_labels]
pred_label_names = [id2label[i] for i in pred_labels]
print(classification_report(true_label_names, pred_label_names))
print("="*80)
 


Evaluating on test set...



TEST SET RESULTS:
Accuracy:  0.7364
Precision: 0.7397
Recall:    0.7364
F1 Score:  0.7354

Per-class F1 scores:
  SUPPORTS: 0.7980
  REFUTES: 0.7420
  NOT ENOUGH INFO: 0.6663

Confusion Matrix:
                                 SUPPORTS         REFUTES NOT ENOUGH INFO
SUPPORTS                             2832             106             395
REFUTES                               283            2288             762
NOT ENOUGH INFO                       650             440            2243

Detailed Classification Report:
                 precision    recall  f1-score   support

NOT ENOUGH INFO       0.66      0.67      0.67      3333
        REFUTES       0.81      0.69      0.74      3333
       SUPPORTS       0.75      0.85      0.80      3333

       accuracy                           0.74      9999
      macro avg       0.74      0.74      0.74      9999
   weighted avg       0.74      0.74      0.74      9999



In [13]:
# ============================================================================
# CELL 13: Save Model
# ============================================================================
 
print("\nSaving model...")
 
output_path = "./stance_detector_final"
os.makedirs(output_path, exist_ok=True)
 
trainer.save_model(output_path)
tokenizer.save_pretrained(output_path)
 
# Save results
results = {
    'test_accuracy':  float(test_results['eval_accuracy']),
    'test_precision': float(test_results['eval_precision']),
    'test_recall':    float(test_results['eval_recall']),
    'test_f1':        float(test_results['eval_f1']),
    'model_name':     config.MODEL_NAME,
    'epochs':         config.NUM_EPOCHS,
    'per_class_f1':   {}
}
 
for label in label_names:
    key = f"eval_f1_{label.lower().replace(' ', '_')}"
    if key in test_results:
        results['per_class_f1'][label] = float(test_results[key])
 
with open(f"{output_path}/results.json", 'w') as f:
    json.dump(results, f, indent=2)
 
# Save label mapping
with open(f"{output_path}/labels.json", 'w') as f:
    json.dump(label_info, f, indent=2)
 
# README
readme = f"""# Stance Detection Model
 
Trained on FEVER-NLI dataset for claim-evidence stance classification.
 
## Model Details
- Base model: {config.MODEL_NAME}
- Epochs: {config.NUM_EPOCHS}
- Training examples: {len(train_df):,}
 
## Performance
- Accuracy:  {test_results['eval_accuracy']:.4f}
- Precision: {test_results['eval_precision']:.4f}
- Recall:    {test_results['eval_recall']:.4f}
- F1 Score:  {test_results['eval_f1']:.4f}
 
## Labels
- LABEL_0: SUPPORTS
- LABEL_1: REFUTES
- LABEL_2: NOT ENOUGH INFO
 
## Usage
```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
 
tokenizer = AutoTokenizer.from_pretrained("./stance_detector_final")
model = AutoModelForSequenceClassification.from_pretrained("./stance_detector_final")
 
claim    = "India's GDP grew 8% in 2024"
evidence = "Official data shows India's economy expanded by 8% in 2024"
 
inputs = tokenizer(claim, evidence, return_tensors="pt", truncation=True, max_length=256)
with torch.no_grad():
    logits = model(**inputs).logits
 
predicted_id = logits.argmax().item()
print(model.config.id2label[predicted_id])
```
"""
 
with open(f"{output_path}/README.md", 'w') as f:
    f.write(readme)
 
print(f"✓ Model saved to: {output_path}")
 


Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model saved to: ./stance_detector_final


In [14]:
# ============================================================================
# CELL 14: Test Saved Model
# ============================================================================
 
print("\nTesting saved model...")
 
from transformers import AutoTokenizer, AutoModelForSequenceClassification
 
saved_tokenizer = AutoTokenizer.from_pretrained(output_path)
saved_model     = AutoModelForSequenceClassification.from_pretrained(output_path)
saved_model.eval()
 
test_cases = [
    {
        'claim':    "India's GDP grew 8% in 2024",
        'evidence': "Official government data shows India's economy expanded by 8% in 2024",
        'expected': "SUPPORTS"
    },
    {
        'claim':    "India's GDP grew 8% in 2024",
        'evidence': "India's GDP fell by 2% in 2024 according to official data",
        'expected': "REFUTES"
    },
    {
        'claim':    "India's GDP grew 8% in 2024",
        'evidence': "The weather was sunny in New Delhi today",
        'expected': "NOT ENOUGH INFO"
    }
]
 
print("\nTest predictions:")
print("="*80)
 
for i, case in enumerate(test_cases, 1):
    # Correct way — pass as sentence pair
    inputs = saved_tokenizer(
        case['claim'],
        case['evidence'],
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
 
    with torch.no_grad():
        logits = saved_model(**inputs).logits
 
    predicted_id    = logits.argmax().item()
    predicted_label = id2label[predicted_id]
    confidence      = torch.softmax(logits, dim=1).max().item()
 
    status = "✓" if predicted_label == case['expected'] else "✗"
 
    print(f"\n{i}. {status}")
    print(f"   Claim:     {case['claim']}")
    print(f"   Evidence:  {case['evidence'][:70]}...")
    print(f"   Expected:  {case['expected']}")
    print(f"   Predicted: {predicted_label}")
    print(f"   Confidence: {confidence:.3f}")
 
print("\n" + "="*80)
print("✓ Model test complete!")
 


Testing saved model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Test predictions:

1. ✓
   Claim:     India's GDP grew 8% in 2024
   Evidence:  Official government data shows India's economy expanded by 8% in 2024...
   Expected:  SUPPORTS
   Predicted: SUPPORTS
   Confidence: 0.722

2. ✗
   Claim:     India's GDP grew 8% in 2024
   Evidence:  India's GDP fell by 2% in 2024 according to official data...
   Expected:  REFUTES
   Predicted: SUPPORTS
   Confidence: 0.570

3. ✓
   Claim:     India's GDP grew 8% in 2024
   Evidence:  The weather was sunny in New Delhi today...
   Expected:  NOT ENOUGH INFO
   Predicted: NOT ENOUGH INFO
   Confidence: 0.722

✓ Model test complete!


In [15]:

# ============================================================================
# CELL 15: Create Download Package
# ============================================================================
 
print("\nCreating download package...")
 
os.system("zip -r stance_detector_final.zip stance_detector_final/")
 
print("✓ Package created: stance_detector_final.zip")
print("\nTo download:")
print("  1. Click on 'stance_detector_final.zip' in the output files panel")
print("  2. Click the download button")
print("  3. Extract on your local machine to:")
print("     fact-verification-system/models/stance_detector/final/")
 


Creating download package...
  adding: stance_detector_final/ (stored 0%)
  adding: stance_detector_final/labels.json (deflated 53%)
  adding: stance_detector_final/results.json (deflated 37%)
  adding: stance_detector_final/config.json (deflated 52%)
  adding: stance_detector_final/README.md (deflated 42%)
  adding: stance_detector_final/training_args.bin (deflated 53%)
  adding: stance_detector_final/tokenizer_config.json (deflated 43%)
  adding: stance_detector_final/model.safetensors (deflated 7%)
  adding: stance_detector_final/tokenizer.json (deflated 70%)
✓ Package created: stance_detector_final.zip

To download:
  1. Click on 'stance_detector_final.zip' in the output files panel
  2. Click the download button
  3. Extract on your local machine to:
     fact-verification-system/models/stance_detector/final/


In [16]:
# ============================================================================
# CELL 16: Summary
# ============================================================================
 
print("\n" + "="*80)
print("TRAINING COMPLETE - SUMMARY")
print("="*80)
 
print(f"\nModel:            {config.MODEL_NAME}")
print(f"Training examples: {len(train_df):,}")
 
print(f"\nFinal Test Results:")
print(f"  Accuracy:  {test_results['eval_accuracy']:.4f}")
print(f"  Precision: {test_results['eval_precision']:.4f}")
print(f"  Recall:    {test_results['eval_recall']:.4f}")
print(f"  F1 Score:  {test_results['eval_f1']:.4f}")
 
print(f"\nPer-class F1:")
for label in label_names:
    key = f"eval_f1_{label.lower().replace(' ', '_')}"
    if key in test_results:
        print(f"  {label}: {test_results[key]:.4f}")
 
print(f"\nDownload file: stance_detector_final.zip")
 
print("\n" + "="*80)
print("NEXT STEPS:")
print("="*80)
print("1. Download stance_detector_final.zip")
print("2. Extract to: models/stance_detector/final/")
print("3. Update configs/nlp_config.yaml:")
print("     stance_detector:")
print("       use_trained: true")
print("="*80)


TRAINING COMPLETE - SUMMARY

Model:            bert-base-cased
Training examples: 208,346

Final Test Results:
  Accuracy:  0.7364
  Precision: 0.7397
  Recall:    0.7364
  F1 Score:  0.7354

Per-class F1:
  SUPPORTS: 0.7980
  REFUTES: 0.7420
  NOT ENOUGH INFO: 0.6663

Download file: stance_detector_final.zip

NEXT STEPS:
1. Download stance_detector_final.zip
2. Extract to: models/stance_detector/final/
3. Update configs/nlp_config.yaml:
     stance_detector:
       use_trained: true
